# USTC-TFC2016 전처리 파이프라인 v2 (메모리 절약)
**캡스톤디자인 I** - 특징 추출 최적화 기반 암호화된 공격 트래픽 탐지 및 분류 시스템
팀장: 김강민 | 팀원: 서은빈

---
### 핵심 변경 (v1 → v2)
- `rdpcap()` (전체 메모리 로드) → `PcapReader()` (패킷 하나씩 스트리밍)
- 파일별로 처리 후 즉시 npy에 append 저장 → 메모리 상시 낮게 유지

### 전처리 흐름
```
pcap 파일 (스트리밍)
  → 태스크 무관 패킷 제거 (ARP, ICMPv6 등)
  → 페이로드 없는 패킷 제거
  → 너무 짧은 패킷 제거 (< 40 bytes)
  → IP/Port 마스킹 (SII 제거 - 숏컷 러닝 방지)
  → 플로우 단위로 묶기
  → 플로우당 첫 784바이트 추출 (28x28 CNN 입력)
  → 파일별 즉시 저장 → 최종 병합
```

## 0. Google Drive 마운트

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE_DIR   = '/content/drive/MyDrive/capstone2026'
USTC_DIR   = os.path.join(BASE_DIR, 'USTC-TFC2016')
OUTPUT_DIR = os.path.join(BASE_DIR, 'output')
TEMP_DIR   = '/content/ustc_temp'  # 임시 파일 로컬 저장 (Drive I/O 줄이기)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(TEMP_DIR, exist_ok=True)

print(f'USTC_DIR  : {USTC_DIR}')
print(f'OUTPUT_DIR: {OUTPUT_DIR}')
print(f'TEMP_DIR  : {TEMP_DIR}')

Mounted at /content/drive
USTC_DIR  : /content/drive/MyDrive/capstone2026/USTC-TFC2016
OUTPUT_DIR: /content/drive/MyDrive/capstone2026/output
TEMP_DIR  : /content/ustc_temp


## 1. 라이브러리 설치 및 임포트

In [3]:
!pip install scapy -q

import os
import gc
import glob
import numpy as np
from collections import defaultdict
from scapy.all import PcapReader, IP, TCP, UDP, Raw

print('완료')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 102.5 MB/s eta 0:00:00
완료


## 2. 설정값

In [3]:
FLOW_BYTES     = 784   # 28x28 CNN 입력
MIN_PKT_LEN    = 40    # 최소 패킷 길이
MIN_FLOW_BYTES = 120   # 최소 플로우 바이트 (MIN_PKT_LEN * 3)
LABEL_BENIGN   = 0
LABEL_MALWARE  = 1

print(f'설정 완료 | 플로우당 바이트: {FLOW_BYTES}')

설정 완료 | 플로우당 바이트: 784


## 3. pcap 파일 경로 수집

In [4]:
def collect_pcap_files(ustc_dir):
    files = {'benign': [], 'malware': []}
    for label, folder in [('benign', 'Benign'), ('malware', 'Malware')]:
        folder_path = os.path.join(ustc_dir, folder)
        pcaps = list(set(
            glob.glob(os.path.join(folder_path, '**/*.pcap'), recursive=True) +
            glob.glob(os.path.join(folder_path, '*.pcap'))
        ))
        files[label] = sorted(pcaps)
        print(f'[{folder}] {len(pcaps)}개')
        for p in pcaps:
            print(f'  {os.path.basename(p)} ({os.path.getsize(p)/1024/1024:.1f} MB)')
    return files

pcap_files = collect_pcap_files(USTC_DIR)

[Benign] 14개
  FTP.pcap (60.2 MB)
  WorldOfWarcraft.pcap (14.9 MB)
  BitTorrent.pcap (7.3 MB)
  SMB-1.pcap (1034.0 MB)
  SMB-2.pcap (206.8 MB)
  Skype.pcap (4.2 MB)
  Weibo-2.pcap (206.8 MB)
  Weibo-3.pcap (206.8 MB)
  Weibo-4.pcap (206.8 MB)
  Outlook.pcap (11.2 MB)
  Gmail.pcap (9.1 MB)
  Facetime.pcap (2.4 MB)
  Weibo-1.pcap (1033.8 MB)
  MySQL.pcap (22.4 MB)
[Malware] 10개
  Geodo.pcap (28.9 MB)
  Nsis-ay.pcap (281.2 MB)
  Tinba.pcap (2.6 MB)
  Htbot.pcap (83.6 MB)
  Zeus.pcap (13.4 MB)
  Miuref.pcap (16.4 MB)
  Shifu.pcap (57.9 MB)
  Neris.pcap (90.1 MB)
  Virut.pcap (109.3 MB)
  Cridex.pcap (94.8 MB)


## 4. 스트리밍 전처리 함수

In [5]:
def extract_flow_key(pkt):
    """양방향 플로우 키 추출 (IP/Port는 키 용도로만, 바이트엔 포함 안 함)"""
    if IP not in pkt:
        return None
    proto = 'TCP' if TCP in pkt else 'UDP' if UDP in pkt else None
    if proto is None:
        return None
    src_ip   = pkt[IP].src
    dst_ip   = pkt[IP].dst
    src_port = pkt[TCP].sport if proto == 'TCP' else pkt[UDP].sport
    dst_port = pkt[TCP].dport if proto == 'TCP' else pkt[UDP].dport
    # 양방향 통일
    if (src_ip, src_port) > (dst_ip, dst_port):
        src_ip, dst_ip = dst_ip, src_ip
        src_port, dst_port = dst_port, src_port
    return (src_ip, dst_ip, src_port, dst_port, proto)


def pcap_to_flows_streaming(pcap_path):
    """PcapReader로 패킷 하나씩 읽어 메모리 절약"""
    flows = defaultdict(bytearray)
    stats = {'total':0, 'no_ip':0, 'no_proto':0,
             'no_payload':0, 'too_short':0, 'valid':0}

    try:
        with PcapReader(pcap_path) as reader:
            for pkt in reader:
                stats['total'] += 1

                # 노이즈 제거
                if IP not in pkt:
                    stats['no_ip'] += 1; continue
                if TCP not in pkt and UDP not in pkt:
                    stats['no_proto'] += 1; continue
                if Raw not in pkt:
                    stats['no_payload'] += 1; continue
                if len(pkt) < MIN_PKT_LEN:
                    stats['too_short'] += 1; continue

                flow_key = extract_flow_key(pkt)
                if flow_key is None:
                    continue

                # 이미 784바이트 다 찼으면 스킵
                if len(flows[flow_key]) >= FLOW_BYTES:
                    continue

                # SII 제거: IP 레이어부터 바이트 추출
                raw_bytes = bytearray(bytes(pkt[IP]))

                # IP src/dst 마스킹 (offset 12~19)
                raw_bytes[12:20] = b'\x00' * 8

                # TCP/UDP port 마스킹 (IP 헤더 뒤 0~3)
                ip_hdr_len = (raw_bytes[0] & 0x0F) * 4
                if len(raw_bytes) > ip_hdr_len + 4:
                    raw_bytes[ip_hdr_len:ip_hdr_len+4] = b'\x00' * 4

                remaining = FLOW_BYTES - len(flows[flow_key])
                flows[flow_key].extend(raw_bytes[:remaining])
                stats['valid'] += 1

    except Exception as e:
        print(f'  오류: {e}')

    return flows, stats


def flows_to_array(flows, label_val):
    """플로우 딕셔너리 → numpy array 변환"""
    X_list, y_list = [], []
    for flow_bytes in flows.values():
        if len(flow_bytes) < MIN_FLOW_BYTES:
            continue
        padded = np.zeros(FLOW_BYTES, dtype=np.float32)
        padded[:len(flow_bytes)] = np.frombuffer(
            bytes(flow_bytes[:FLOW_BYTES]), dtype=np.uint8
        ).astype(np.float32) / 255.0
        X_list.append(padded)
        y_list.append(label_val)
    return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.int32)

print('함수 정의 완료')

함수 정의 완료


## 5. 전체 처리 (파일별 스트리밍 → 즉시 저장)

In [6]:
all_X, all_y = [], []
class_counts = {}

for label_name, label_val in [('benign', LABEL_BENIGN), ('malware', LABEL_MALWARE)]:
    print(f'\n[{label_name.upper()}] 처리 시작')

    for pcap_path in pcap_files[label_name]:
        class_name = os.path.basename(pcap_path).replace('.pcap', '')
        size_mb = os.path.getsize(pcap_path) / 1024 / 1024
        print(f'  처리중: {class_name} ({size_mb:.1f} MB)...')

        # 스트리밍 처리
        flows, stats = pcap_to_flows_streaming(pcap_path)

        # numpy 변환
        X_file, y_file = flows_to_array(flows, label_val)

        # 메모리 해제
        del flows
        gc.collect()

        if len(X_file) > 0:
            all_X.append(X_file)
            all_y.append(y_file)
            class_counts[class_name] = len(X_file)

        print(f'    → {len(X_file):,}개 플로우 '
              f'| 총:{stats["total"]:,} '
              f'IP없음:{stats["no_ip"]:,} '
              f'페이로드없음:{stats["no_payload"]:,} '
              f'유효:{stats["valid"]:,}')

# 병합
X = np.concatenate(all_X, axis=0)
y = np.concatenate(all_y, axis=0)
del all_X, all_y
gc.collect()

print(f'\n[완료]')
print(f'  X shape : {X.shape}')
print(f'  y shape : {y.shape}')
print(f'  Benign  : {(y==0).sum():,}개')
print(f'  Malware : {(y==1).sum():,}개')
print(f'\n클래스별 플로우 수:')
for cls, cnt in class_counts.items():
    print(f'  {cls}: {cnt:,}')


[BENIGN] 처리 시작
  처리중: BitTorrent (7.3 MB)...
    → 7,502개 플로우 | 총:15,000 IP없음:0 페이로드없음:7,498 유효:7,502
  처리중: FTP (60.2 MB)...
    → 26,424개 플로우 | 총:360,000 IP없음:0 페이로드없음:236,069 유효:111,447
  처리중: Facetime (2.4 MB)...
    → 6,000개 플로우 | 총:6,000 IP없음:0 페이로드없음:0 유효:6,000
  처리중: Gmail (9.1 MB)...
    → 6,442개 플로우 | 총:25,000 IP없음:0 페이로드없음:17,471 유효:6,442
  처리중: MySQL (22.4 MB)...
    → 21,400개 플로우 | 총:200,000 IP없음:0 페이로드없음:127,912 유효:72,088
  처리중: Outlook (11.2 MB)...
    → 7,475개 플로우 | 총:15,000 IP없음:0 페이로드없음:7,525 유효:7,475
  처리중: SMB-1 (1034.0 MB)...
    → 29,231개 플로우 | 총:771,887 IP없음:1 페이로드없음:52,238 유효:31,755
  처리중: SMB-2 (206.8 MB)...
    → 5,759개 플로우 | 총:153,566 IP없음:0 페이로드없음:9,730 유효:6,200
  처리중: Skype (4.2 MB)...
    → 6,089개 플로우 | 총:12,000 IP없음:0 페이로드없음:5,911 유효:6,089
  처리중: Weibo-1 (1033.8 MB)...
    → 24,908개 플로우 | 총:756,068 IP없음:1 페이로드없음:40,767 유효:24,908
  처리중: Weibo-2 (206.8 MB)...
    → 5,033개 플로우 | 총:151,674 IP없음:0 페이로드없음:8,589 유효:5,033
  처리중: Weibo-3 (206.8 MB)...
    → 5,006

## 6. 최종 검증

In [7]:
print('[최종 검증]')

ip_masked  = (X[:, 12:20] == 0).all()
nan_count  = int(np.isnan(X).sum())
inf_count  = int(np.isinf(X).sum())
val_ok     = float(X.min()) >= 0 and float(X.max()) <= 1

print(f'  IP 마스킹 : {"✅" if ip_masked else "❌"}')
print(f'  NaN       : {nan_count}개 {"✅" if nan_count == 0 else "❌"}')
print(f'  Inf       : {inf_count}개 {"✅" if inf_count == 0 else "❌"}')
print(f'  값 범위   : [{float(X.min()):.3f}, {float(X.max()):.3f}] {"✅" if val_ok else "❌"}')
print(f'\n[클래스 분포]')
print(f'  Benign  : {(y==0).sum():,} ({(y==0).mean()*100:.1f}%)')
print(f'  Malware : {(y==1).sum():,} ({(y==1).mean()*100:.1f}%)')

[최종 검증]
  IP 마스킹 : ✅
  NaN       : 0개 ✅
  Inf       : 0개 ✅
  값 범위   : [0.000, 1.000] ✅

[클래스 분포]
  Benign  : 164,035 (80.5%)
  Malware : 39,683 (19.5%)


## 7. 저장

In [8]:
X_path = os.path.join(OUTPUT_DIR, 'ustc_X.npy')
y_path = os.path.join(OUTPUT_DIR, 'ustc_y.npy')

np.save(X_path, X)
np.save(y_path, y)

print(f'[저장 완료]')
print(f'  X : {X_path}  ({os.path.getsize(X_path)/1024/1024:.1f} MB)')
print(f'  y : {y_path}  ({os.path.getsize(y_path)/1024/1024:.1f} MB)')

[저장 완료]
  X : /content/drive/MyDrive/capstone2026/output/ustc_X.npy  (609.3 MB)
  y : /content/drive/MyDrive/capstone2026/output/ustc_y.npy  (0.8 MB)
